In [5]:
#!/usr/bin/env python3
# mc_rho_vs_beta.py
"""
MC sweep: for each kappa in KAPPAS and for each beta in BETAS,
 - generate R independent random 3-uniform hypergraphs H(n,m,3)
 - run stochastic MC (s-SIS with qi computed from EMA-smoothed p and Theta)
 - collect final rho (at T steps) per run -> compute mean/std across runs
 - save summary CSV and runs-matrix CSV for each kappa

Notes:
 - We use p_smooth (EMA) so MC per-step infection prob matches MMCA-style qi.
 - This script is adapted from your provided MC implementation.
"""
import os, math, time, random
import numpy as np
import pandas as pd



# reuse hypergraph generator from mmca script
def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    rng = np.random.RandomState(seed)
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = rng.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = np.random.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

class HyperMC:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        self.edge_i = self.hyperedges[:,0]; self.edge_j = self.hyperedges[:,1]; self.edge_k = self.hyperedges[:,2]
        self.node_edge_pos = [[] for _ in range(n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def compute_qi(self, p, Theta, beta_d):
        # vectorized for three positions
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        prod_pos0 = p_j * p_k
        prod_pos1 = p_i * p_k
        prod_pos2 = p_i * p_j
        qi = np.zeros(self.n, dtype=float)
        for node in range(self.n):
            pv = 1.0
            for (ei,pos) in self.node_edge_pos[node]:
                if pos == 0:
                    oth = prod_pos0[ei]
                elif pos == 1:
                    oth = prod_pos1[ei]
                else:
                    oth = prod_pos2[ei]
                pv *= (1.0 - Theta[ei] * beta_d * oth)
            qi[node] = 1.0 - pv
        return np.clip(qi, 0.0, 1.0)

    def update_theta_from_p(self, p, Theta, eta, gamma):
        p_i = p[self.edge_i]; p_j = p[self.edge_j]; p_k = p[self.edge_k]
        Phi = p_i * p_j * p_k
        Theta = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
        Theta = np.clip(Theta, 0.0, 1.0)
        return Theta

    def run_mc(self, beta_d, mu, eta, gamma, T, init_frac, p_smooth, seed=None, final_window=20):
        
        rng = np.random.RandomState(seed)
        
        states = np.zeros(self.n, dtype=int)
        initial_infected = max(1, int(round(init_frac * self.n)))
        init_nodes = rng.choice(self.n, size=initial_infected, replace=False)
        states[init_nodes] = 1
        p = np.full(self.n, init_frac, dtype=float)
        Theta = np.ones(self.m, dtype=float)
        # 记录最后窗口（循环内可按滚动缓冲来节省内存）
        last_window = []

        for t in range(T):
            # EMA 更新 p
            if p_smooth == 0.0:
                p = states.astype(float)
            else:
                alpha = float(p_smooth)
                p = alpha * p + (1.0 - alpha) * states

            qi = self.compute_qi(p, Theta, beta_d)

            # infection trials (vectorized)
            sus_idx = np.where(states == 0)[0]
            if sus_idx.size > 0:
                rand_vals = rng.random(sus_idx.size)
                hit_mask = rand_vals < qi[sus_idx]
                new_infections = sus_idx[hit_mask]
            else:
                new_infections = np.array([], dtype=int)

            # recovery trials
            inf_idx = np.where(states == 1)[0]
            if inf_idx.size > 0:
                rec_rand = rng.random(inf_idx.size)
                recovered = inf_idx[rec_rand < mu]
            else:
                recovered = np.array([], dtype=int)

            # sync update: recover first then infections
            next_states = states.copy()
            if recovered.size > 0:
                next_states[recovered] = 0
            if new_infections.size > 0:
                # 直接赋值比循环更快
                next_states[new_infections] = 1
            states = next_states

            # update Theta from p (use p computed at this step)
            Theta = self.update_theta_from_p(p, Theta, eta, gamma)

            # record last window
            last_window.append(states.mean())
            if len(last_window) > final_window:
                last_window.pop(0)

        # 返回最后窗口的平均与 std，便于后续统计
        final_mean = float(np.mean(last_window)) if len(last_window) > 0 else float(states.mean())
        
        return final_mean
    
# -------------------- Parameters --------------------
N = 1500               # node count
KAPPAS = [6.0, 9.0, 12.0]
R = 100
BETAS = np.concatenate([
    np.arange(0.00, 0.1, 0.02),
    np.arange(0.1, 0.2, 0.02),
    np.arange(0.2, 0.3, 0.05),
    np.arange(0.3, 0.7, 0.1)
])
MU = 0.1
ETA = 0.6
GAMMA = 0.02
INIT_FRAC = 0.3
T = 300               # MC time steps per run (makes final-state approximate steady)
P_SMOOTH = 0.0       # EMA smoothing to align MC with MMCA (0..1)
SEED_BASE = 12345

OUT_DIR = "out_mc"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------- sweep --------------------
t0 = time.time()
for kappa in KAPPAS:
    print(f"\n=== MC: kappa={kappa}  R={R}  betas={len(BETAS)} ===")
    runs_matrix = np.zeros((len(BETAS), R), dtype=float)
    for ib, beta in enumerate(BETAS):
        for r in range(R):
            gen_seed = SEED_BASE  + r
            hyperedges = generate_hypergraph_H_n_m_3(N, kappa, seed=gen_seed)
            model = HyperMC(hyperedges, N)
            run_seed = gen_seed + 1000 + r
            rho_final = model.run_mc(beta_d=beta, mu=MU, eta=ETA, gamma=GAMMA,
                                     T=T, init_frac=INIT_FRAC, p_smooth=P_SMOOTH, seed=run_seed, final_window=20)
            runs_matrix[ib, r] = rho_final
        mean_rho = runs_matrix[ib].mean()
        std_rho = runs_matrix[ib].std(ddof=0)
        print(f"k={kappa} beta={beta:.3f}  mean_rho={mean_rho:.4f} std={std_rho:.4f}")
    # save
    summary_df = pd.DataFrame({
        'beta': BETAS,
        'mean_rho': runs_matrix.mean(axis=1),
        'std_rho': runs_matrix.std(axis=1, ddof=0)
    })
    runs_df = pd.DataFrame(
        runs_matrix, 
        columns=[f'run{r:03d}' for r in range(R)]
    )

    summary_fn = os.path.join(OUT_DIR, f"mc_kappa{int(kappa)}_summary.csv")
    runs_fn = os.path.join(OUT_DIR, f"mc_kappa{int(kappa)}_runs.csv")
    summary_df.to_csv(summary_fn, index=False, float_format="%.4f")
    runs_df.to_csv(runs_fn, index=False, float_format="%.4f")
    print(f"Saved {summary_fn} and {runs_fn}")

print("All done. Total time: {:.1f}s".format(time.time()-t0))



=== MC: kappa=6.0  R=100  betas=16 ===
k=6.0 beta=0.000  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.020  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.040  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.060  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.080  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.100  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.120  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.140  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.160  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.180  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.200  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.250  mean_rho=0.2512 std=0.2034
k=6.0 beta=0.300  mean_rho=0.4882 std=0.0189
k=6.0 beta=0.400  mean_rho=0.5591 std=0.0125
k=6.0 beta=0.500  mean_rho=0.6059 std=0.0121
k=6.0 beta=0.600  mean_rho=0.6395 std=0.0076
Saved out_mc\mc_kappa6_summary.csv and out_mc\mc_kappa6_runs.csv

=== MC: kappa=9.0  R=100  betas=16 ===
k=9.0 beta=0.000  mean_rho=0.0000 std=0.0000
k=9.0 beta=0.020  mean_rho=0.0000 std=0.0000
k=9.0 beta=0.040  mean_rho=0.0000 std=0.0000
